In [33]:
import os
import cv2
import base64
import numpy as np
import fitz  # pymupdf
from pathlib import Path
from typing import Optional, Literal
from dotenv import load_dotenv
from pydantic import BaseModel
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

load_dotenv(dotenv_path=os.path.join("..", ".env"), override=True)

llm = ChatOpenAI(
    model=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    base_url=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    temperature=0,       # zero temp — extraction must be deterministic
    timeout=60,
    max_retries=2,
)

# sanity check
print(llm.invoke("Reply with: InBody parser online").content)

InBody parser online


In [34]:
class SegmentalReading(BaseModel):
    value: float
    unit: Literal["kg", "lb"]
    percent_of_ideal: Optional[float] = None  # the % bar shown on the scan


class InBodyRawExtraction(BaseModel):
    # ── Identity ─────────────────────────────────────────────────────────────
    inbody_model: Optional[str] = None          # "270S" / "570" / "770"
    gender: Literal["male", "female"]
    age: Optional[int] = None

    # ── Core composition ─────────────────────────────────────────────────────
    weight: float
    weight_unit: Literal["kg", "lb"]
    skeletal_muscle_mass: float
    smm_unit: Literal["kg", "lb"]
    body_fat_percent: float                     # PBF — always a plain % number
    bmr_kcal: Optional[int] = None             # not present on 270S

    # ── Segmental lean ───────────────────────────────────────────────────────
    right_arm: SegmentalReading
    left_arm:  SegmentalReading
    trunk:     SegmentalReading
    right_leg: SegmentalReading
    left_leg:  SegmentalReading

    # ── Higher-model only (570 / 770) ────────────────────────────────────────
    ecw_ratio: Optional[float] = None
    visceral_fat_level: Optional[int] = None    # 570: level 1-20
    visceral_fat_area_cm2: Optional[float] = None  # 770: cm²
    smi: Optional[float] = None                 # Skeletal Muscle Index

    # ── Recovery indicators (present on select models) ────────────────────
    phase_angle: Optional[float] = None         # 270S + 770: cell health indicator
                                                # low (<5.0) = poor recovery capacity
    waist_hip_ratio: Optional[float] = None     # 570 + 770: fat distribution pattern
                                                # high (>0.90 male / >0.85 female) = central obesity

    extraction_notes: Optional[str] = None      # anything the VLM couldn't read clearly


class InBodyFlags(BaseModel):
    arm_asymmetry: bool
    arm_diff_grams: float
    weaker_arm: Optional[Literal["left", "right"]] = None

    leg_asymmetry: bool
    leg_diff_grams: float
    weaker_leg: Optional[Literal["left", "right"]] = None

    elevated_bf: bool
    trunk_underdeveloped: bool


class InBodyResult(BaseModel):
    raw: InBodyRawExtraction
    flags: InBodyFlags


# ── Validation model (Stage 2 lightweight check) ─────────────────────────────
class InBodyValidation(BaseModel):
    is_inbody_scan: bool
    confidence: Literal["high", "medium", "low"]
    issue: Optional[str] = None   # e.g. "appears to be a food label, not an InBody scan"


print("Models defined.")

Models defined.


In [35]:
%pwd

'e:\\vs codes\\Tamreena_AI\\notebooks'

In [24]:
def check_image_quality(image_bytes: bytes) -> dict:
    """
    Checks blur and brightness from raw image bytes.
    Returns {"pass": bool, "issue": str | None, "blur_score": float, "brightness": float}
    """
    arr = np.frombuffer(image_bytes, np.uint8)
    img = cv2.imdecode(arr, cv2.IMREAD_COLOR)

    if img is None:
        return {"pass": False, "issue": "Could not decode image.", "blur_score": 0, "brightness": 0}

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    blur_score = cv2.Laplacian(gray, cv2.CV_64F).var()
    brightness = float(gray.mean())

    if blur_score < 100:
        return {
            "pass": False,
            "issue": f"Image is too blurry (score: {blur_score:.1f}). Retake in better lighting or hold the camera steady.",
            "blur_score": blur_score,
            "brightness": brightness,
        }
    if brightness < 50:
        return {
            "pass": False,
            "issue": "Image is too dark. Move to a brighter area or turn on a light.",
            "blur_score": blur_score,
            "brightness": brightness,
        }
    dark_pixel_ratio = float((gray < 80).mean())
    if dark_pixel_ratio < 0.02:
        return {
            "pass": False,
            "issue": "Image is overexposed — no text is visible. Move away from direct light.",
            "blur_score": blur_score,
            "brightness": brightness,
        }

    return {"pass": True, "issue": None, "blur_score": blur_score, "brightness": brightness}


# ── Test ─────────────────────────────────────────────────────────────────────
# Replace with a path to one of your InBody scan images
TEST_IMAGE_PATH = "../samples/inbody2.jfif"

with open(TEST_IMAGE_PATH, "rb") as f:
    image_bytes = f.read()

quality_result = check_image_quality(image_bytes)
print(quality_result)

{'pass': False, 'issue': 'Image is overexposed — no text is visible. Move away from direct light.', 'blur_score': np.float64(4383.055110677084), 'brightness': 237.16902018229166}


In [25]:
def pdf_to_image_bytes(pdf_bytes: bytes, zoom: float = 2.0) -> bytes:
    """
    Renders the first page of a PDF to PNG bytes.
    zoom=2.0 doubles the resolution — important for small text on InBody scans.
    """
    doc = fitz.open(stream=pdf_bytes, filetype="pdf")
    page = doc[0]
    mat = fitz.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat)
    return pix.tobytes("png")


def load_scan(file_path: str) -> tuple[bytes, str]:
    """
    Loads an InBody scan from a file path.
    Returns (image_bytes, content_type).
    PDF is converted to PNG automatically.
    """
    path = Path(file_path)
    raw = path.read_bytes()

    if path.suffix.lower() == ".pdf":
        image_bytes = pdf_to_image_bytes(raw)
        return image_bytes, "image/png"

    ext_to_type = {".jpg": "image/jpeg", ".jpeg": "image/jpeg", ".png": "image/png"}
    content_type = ext_to_type.get(path.suffix.lower(), "image/jpeg")
    return raw, content_type


# ── Test ─────────────────────────────────────────────────────────────────────
# Uncomment to test PDF conversion
# pdf_bytes, ct = load_scan("samples/inbody_scan.pdf")
# print(f"Converted PDF → {ct}, {len(pdf_bytes)} bytes")

In [26]:
VALIDATION_PROMPT = """You are checking whether an image is an InBody body composition result sheet.

Look for these identifiers:
- InBody logo in the top-left corner
- "Body Composition Analysis" section
- "Segmental Lean Analysis" section with arm/leg/trunk values
- Typical InBody layout with measurement bars or tables

Return your assessment. If it is NOT an InBody scan, describe what it appears to be instead."""

def validate_inbody_scan(image_bytes: bytes, content_type: str) -> InBodyValidation:
    """
    Lightweight LLM check — runs BEFORE the full extraction pipeline.
    Returns InBodyValidation with is_inbody_scan bool and optional issue message.
    """
    b64 = base64.b64encode(image_bytes).decode("utf-8")
    validation_llm = llm.with_structured_output(InBodyValidation)

    result = validation_llm.invoke([
        SystemMessage(content=VALIDATION_PROMPT),
        HumanMessage(content=[
            {"type": "image_url", "image_url": {"url": f"data:{content_type};base64,{b64}"}},
            {"type": "text", "text": "Is this an InBody body composition result sheet?"},
        ]),
    ])
    return result


# ── Test ─────────────────────────────────────────────────────────────────────
image_bytes, content_type = load_scan(TEST_IMAGE_PATH)
validation = validate_inbody_scan(image_bytes, content_type)
print(validation)
# Expected: InBodyValidation(is_inbody_scan=True, confidence='high', issue=None)

is_inbody_scan=True confidence='high' issue=None


In [36]:
EXTRACTION_PROMPT = """You are an InBody scan data extractor.

Your ONLY job is to read values printed on the scan and return them in the exact schema.

Rules:
- NEVER calculate, derive, or infer values — only read what is explicitly printed
- If a field is not on this InBody model or you cannot read it clearly → return null
- ALWAYS extract the unit (kg or lb) alongside every measurement — do not assume
- If a value is partially obscured or ambiguous, set it to null and note it in extraction_notes
- Do not guess. A null is always better than a wrong number.

── body_fat_percent ────────────────────────────────────────────────────────────
This is the PBF (Percent Body Fat) value — a plain percentage number (e.g. 35.0, not 0.35).
It is found in the Obesity Analysis section, labeled "PBF" or "Percent Body Fat".
CRITICAL: Do NOT use BMI — BMI is a different field (kg/m²) and will always be wrong here.
CRITICAL: Do NOT use Body Fat Mass (the weight value in kg or lb) — that is a different field.
Example: Obesity Analysis shows "BMI: 22.1" and "PBF: 35.0" → extract 35.0, not 22.1.

── Segmental Lean Analysis values ──────────────────────────────────────────────
Read the number printed in text next to each segment label (Right Arm, Left Arm, Trunk,
Right Leg, Left Leg). Each segment has its weight value printed explicitly as a number with
a unit (kg or lb).
CRITICAL: Do NOT estimate values from bar length — the bars are visual only.
CRITICAL: Read each segment's value independently — never copy one segment's value to another.
The percent_of_ideal is the percentage number printed next to or below each segment's bar
(e.g. 139.0). Read it from the same row as the segment's weight value.

── bmr_kcal ────────────────────────────────────────────────────────────────────
Found in the "Research Parameters" section, labeled "Basal Metabolic Rate".
This section is often on the RIGHT side of the scan.
Present on 270S, 570, and 770 — check the Research Parameters section before returning null.

── Other field locations ────────────────────────────────────────────────────────
- inbody_model   → top of the scan in brackets, e.g. "[InBody270S]" → extract "270S"
- skeletal_muscle_mass / smm_unit → "Muscle-Fat Analysis" section, "SMM" row
- ecw_ratio      → "ECW Ratio" or "ECW/TBW" section (570/770 only)
- visceral_fat_level    → "Visceral Fat Level" scale 1–20 (570 only)
- visceral_fat_area_cm2 → "VFA" in cm² (770 only)
- smi            → "SMI" in Research Parameters, unit kg/m² (570/770 only)

── phase_angle ─────────────────────────────────────────────────────────────────
Found on 270S and 770, labeled "Whole Body Phase Angle" or "Phase Angle".
Extract the numeric value only (e.g. 7.5, not "7.5°").
Location: often in a small box in the top-right area or Research Parameters section.
If not visible or not on this model → return null.

── waist_hip_ratio ─────────────────────────────────────────────────────────────
Found on 570 and 770 in the Research Parameters section, labeled "Waist-Hip Ratio".
Extract the numeric value only (e.g. 0.83).
If not visible or not on this model → return null."""


def extract_inbody(image_bytes: bytes, content_type: str) -> InBodyRawExtraction:
    """
    Sends the scan image to GPT-4.1-mini vision with structured output.
    Returns InBodyRawExtraction — raw numbers only, no flags.
    """
    b64 = base64.b64encode(image_bytes).decode("utf-8")
    extraction_llm = llm.with_structured_output(InBodyRawExtraction)

    result = extraction_llm.invoke([
        SystemMessage(content=EXTRACTION_PROMPT),
        HumanMessage(content=[
            {"type": "image_url", "image_url": {"url": f"data:{content_type};base64,{b64}"}},
            {"type": "text", "text": "Extract all InBody data from this scan."},
        ]),
    ])
    return result


# ── Test ─────────────────────────────────────────────────────────────────────
raw = extract_inbody(image_bytes, content_type)
print(raw.model_dump_json(indent=2))

{
  "inbody_model": "770",
  "gender": "male",
  "age": 41,
  "weight": 66.4,
  "weight_unit": "kg",
  "skeletal_muscle_mass": 26.7,
  "smm_unit": "kg",
  "body_fat_percent": 27.2,
  "bmr_kcal": null,
  "right_arm": {
    "value": 3.75,
    "unit": "kg",
    "percent_of_ideal": 109.6
  },
  "left_arm": {
    "value": 3.75,
    "unit": "kg",
    "percent_of_ideal": 109.5
  },
  "trunk": {
    "value": 26.3,
    "unit": "kg",
    "percent_of_ideal": 105.3
  },
  "right_leg": {
    "value": 10.7,
    "unit": "kg",
    "percent_of_ideal": 109.6
  },
  "left_leg": {
    "value": 10.6,
    "unit": "kg",
    "percent_of_ideal": 109.5
  },
  "ecw_ratio": 0.38,
  "visceral_fat_level": null,
  "visceral_fat_area_cm2": 71.2,
  "smi": null,
  "phase_angle": 6.0,
  "waist_hip_ratio": null,
  "extraction_notes": null
}


In [37]:
def to_kg(value: float, unit: str) -> float:
    """Normalize any segmental value to kg."""
    return value * 0.453592 if unit == "lb" else value


def compute_flags(r: InBodyRawExtraction) -> InBodyFlags:
    """
    Derives all flags deterministically from raw extracted values.
    No LLM involved — no hallucination possible.
    """
    ra_kg = to_kg(r.right_arm.value, r.right_arm.unit)
    la_kg = to_kg(r.left_arm.value,  r.left_arm.unit)
    rl_kg = to_kg(r.right_leg.value, r.right_leg.unit)
    ll_kg = to_kg(r.left_leg.value,  r.left_leg.unit)

    arm_diff_g = abs(ra_kg - la_kg) * 1000
    leg_diff_g = abs(rl_kg - ll_kg) * 1000

    bf_threshold = 18.0 if r.gender == "male" else 25.0

    # trunk: use percent_of_ideal if available — InBody already normalises by body stats
    trunk_pct = r.trunk.percent_of_ideal
    trunk_underdeveloped = (trunk_pct < 100.0) if trunk_pct is not None else False

    return InBodyFlags(
        arm_asymmetry=arm_diff_g > 200,
        arm_diff_grams=round(arm_diff_g, 1),
        weaker_arm=("left" if la_kg < ra_kg else "right" if ra_kg < la_kg else None),

        leg_asymmetry=leg_diff_g > 500,
        leg_diff_grams=round(leg_diff_g, 1),
        weaker_leg=("left" if ll_kg < rl_kg else "right" if rl_kg < ll_kg else None),

        elevated_bf=r.body_fat_percent > bf_threshold,
        trunk_underdeveloped=trunk_underdeveloped,
    )


# ── Test ─────────────────────────────────────────────────────────────────────
flags = compute_flags(raw)
print(flags.model_dump_json(indent=2))

{
  "arm_asymmetry": false,
  "arm_diff_grams": 0.0,
  "weaker_arm": null,
  "leg_asymmetry": false,
  "leg_diff_grams": 100.0,
  "weaker_leg": "left",
  "elevated_bf": true,
  "trunk_underdeveloped": false
}


In [38]:
def run_inbody_pipeline_from_bytes(image_bytes: bytes, content_type: str = "image/png") -> InBodyResult | dict:
    """
    Full pipeline from raw image bytes:
      quality check → authenticity check → extraction → flag computation
    Returns InBodyResult on success, or {"error": str, "stage": str} on failure.
    """
    # Stage 1: image quality
    quality = check_image_quality(image_bytes)
    if not quality["pass"]:
        return {
            "error": quality["issue"],
            "stage": "quality_check",
            "blur_score": quality["blur_score"],
            "brightness": quality["brightness"],
        }

    # Stage 2: authenticity
    validation = validate_inbody_scan(image_bytes, content_type)
    if not validation.is_inbody_scan:
        msg = validation.issue or "This does not appear to be an InBody scan."
        return {"error": msg, "stage": "authenticity_check"}

    # Stage 3: extraction
    raw = extract_inbody(image_bytes, content_type)

    # Stage 4: flag computation
    flags = compute_flags(raw)

    return InBodyResult(raw=raw, flags=flags)


def run_inbody_pipeline(file_path: str) -> InBodyResult | dict:
    """Convenience wrapper — loads from file path then calls the bytes pipeline."""
    image_bytes, content_type = load_scan(file_path)
    return run_inbody_pipeline_from_bytes(image_bytes, content_type)

In [39]:
import time
import threading
import ipywidgets as widgets
from IPython.display import display


class InBodyCameraCapture:
    """
    Live webcam feed inside the notebook.
    Shows an alignment guide overlay so you know where to hold the scan.
    Capture button freezes the frame and feeds it straight into the pipeline.
    """

    def __init__(self, camera_index: int = 0):
        self.camera_index = camera_index
        self.captured_frame = None
        self._last_frame = None
        self.running = False

        # ── widgets ──────────────────────────────────────────────────────────
        self.image_widget = widgets.Image(
            format="jpeg",
            layout=widgets.Layout(width="640px", height="480px"),
        )
        self.capture_btn = widgets.Button(
            description="📸  Capture",
            button_style="success",
            layout=widgets.Layout(width="160px", height="40px"),
        )
        self.status = widgets.HTML(
            value="<b>Hold the InBody scan inside the green frame. Click Capture when clear.</b>"
        )
        self.capture_btn.on_click(self._on_capture)

    # ── guide overlay ─────────────────────────────────────────────────────────
    def _draw_guide(self, frame):
        h, w = frame.shape[:2]
        x1, y1 = int(w * 0.05), int(h * 0.05)
        x2, y2 = int(w * 0.95), int(h * 0.95)
        # outer guide rectangle
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 220, 0), 3)
        # corner accents (L-shaped corners for better visual)
        corner = 30
        for cx, cy, dx, dy in [(x1,y1,1,1),(x2,y1,-1,1),(x1,y2,1,-1),(x2,y2,-1,-1)]:
            cv2.line(frame, (cx, cy), (cx + dx*corner, cy), (0, 255, 0), 4)
            cv2.line(frame, (cx, cy), (cx, cy + dy*corner), (0, 255, 0), 4)
        # instruction text
        cv2.putText(
            frame, "Align InBody scan inside the frame",
            (x1 + 8, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 220, 0), 2,
        )
        return frame

    # ── camera thread ─────────────────────────────────────────────────────────
    def _feed_thread(self):
        cap = cv2.VideoCapture(self.camera_index)
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
        self.running = True

        while self.running:
            ret, frame = cap.read()
            if ret:
                self._last_frame = frame.copy()                    # clean frame for capture
                display_frame = self._draw_guide(frame.copy())     # guide on display copy
                _, buf = cv2.imencode(
                    ".jpg", display_frame, [cv2.IMWRITE_JPEG_QUALITY, 85]
                )
                self.image_widget.value = buf.tobytes()
            time.sleep(0.033)   # ~30 fps

        cap.release()

    # ── capture ───────────────────────────────────────────────────────────────
    def _on_capture(self, _btn):
        if self._last_frame is not None:
            self.captured_frame = self._last_frame.copy()
            self.running = False
            self.capture_btn.disabled = True
            self.status.value = "<b style='color:green'>✅ Captured — running pipeline...</b>"

    # ── public ────────────────────────────────────────────────────────────────
    def start(self):
        """Display the widget and start the live feed."""
        t = threading.Thread(target=self._feed_thread, daemon=True)
        t.start()
        display(widgets.VBox([
            self.status,
            self.image_widget,
            self.capture_btn,
        ]))

    def wait_for_capture(self, timeout: int = 120) -> np.ndarray | None:
        """
        Blocks until the user clicks Capture or timeout is reached.
        Returns the raw captured frame (no guide overlay).
        """
        deadline = time.time() + timeout
        while self.captured_frame is None:
            if time.time() > deadline:
                self.running = False
                return None
            time.sleep(0.1)
        return self.captured_frame


print("InBodyCameraCapture defined.")

InBodyCameraCapture defined.


In [40]:
# 1. Open the live camera feed
camera = InBodyCameraCapture(camera_index=0)  # change index if you have multiple cameras
camera.start()

# 2. Block until you click Capture (or 2 minutes pass)
frame = camera.wait_for_capture(timeout=120)

if frame is None:
    print("⏱ Timed out — no capture made.")
else:
    # 3. Convert captured frame to PNG bytes
    _, buf = cv2.imencode(".png", frame)
    image_bytes = buf.tobytes()

    # 4. Run full pipeline
    result = run_inbody_pipeline_from_bytes(image_bytes, content_type="image/png")

    print("\n" + "="*60)
    if isinstance(result, dict):
        stage = result["stage"]
        error = result["error"]
        print(f"❌  Failed at stage [{stage}]")
        print(f"    → {error}")

        if stage == "quality_check":
            print(f"    Blur score : {result['blur_score']:.1f}  (need > 100)")
            print(f"    Brightness : {result['brightness']:.1f}  (need 50–220)")
    else:
        print("✅  Extraction complete!")
        print(f"\n── Raw ──────────────────────────────────")
        print(f"  Model  : {result.raw.inbody_model}")
        print(f"  Gender : {result.raw.gender}  |  Age: {result.raw.age}")
        print(f"  SMM    : {result.raw.skeletal_muscle_mass} {result.raw.smm_unit}")
        print(f"  BF%    : {result.raw.body_fat_percent}%")
        print(f"  BMR    : {result.raw.bmr_kcal or 'N/A'} kcal")
        print(f"  R.Arm  : {result.raw.right_arm.value} {result.raw.right_arm.unit} ({result.raw.right_arm.percent_of_ideal}%)")
        print(f"  L.Arm  : {result.raw.left_arm.value} {result.raw.left_arm.unit} ({result.raw.left_arm.percent_of_ideal}%)")
        print(f"  Trunk  : {result.raw.trunk.value} {result.raw.trunk.unit} ({result.raw.trunk.percent_of_ideal}%)")
        print(f"  R.Leg  : {result.raw.right_leg.value} {result.raw.right_leg.unit}")
        print(f"  L.Leg  : {result.raw.left_leg.value} {result.raw.left_leg.unit}")
        if result.raw.extraction_notes:
            print(f"  Notes  : {result.raw.extraction_notes}")

        print(f"\n── Flags ────────────────────────────────")
        print(f"  ARM_ASYMMETRY       : {result.flags.arm_asymmetry}"
              f"  (diff {result.flags.arm_diff_grams}g, weaker: {result.flags.weaker_arm})")
        print(f"  LEG_ASYMMETRY       : {result.flags.leg_asymmetry}"
              f"  (diff {result.flags.leg_diff_grams}g, weaker: {result.flags.weaker_leg})")
        print(f"  ELEVATED_BF         : {result.flags.elevated_bf}")
        print(f"  TRUNK_UNDERDEVELOPED: {result.flags.trunk_underdeveloped}")

KeyboardInterrupt: 

In [42]:
# Place your sample scans in a /samples folder next to the notebook.
# Download official samples from:
#   270S: https://fr.inbody.com/wp-content/uploads/2025/02/Example-Result-Sheet_270S-Body-Composition-ENG_240731_out.pdf
#   Professional guide (770): https://uk.inbody.com/wp-content/uploads/2018/08/The_Professional_s_Guide_to_the_InBody_Result_Sheet.pdf

SAMPLES = [
    "../samples/inbody2.jfif",
    "../samples/inbody3.jfif",
    "../samples/inbody4.jfif",
]

for path in SAMPLES:
    if not Path(path).exists():
        print(f"SKIP — file not found: {path}")
        continue

    print(f"\n{'='*60}")
    print(f"Testing: {path}")
    print('='*60)

    result = run_inbody_pipeline(path)

    if isinstance(result, dict):
        print(f"FAILED at stage [{result['stage']}]: {result['error']}")
        continue

    print("\n── Raw extraction ──")
    print(f"  Model    : {result.raw.inbody_model}")
    print(f"  Gender   : {result.raw.gender}")
    print(f"  SMM      : {result.raw.skeletal_muscle_mass} {result.raw.smm_unit}")
    print(f"  BF%      : {result.raw.body_fat_percent}%")
    print(f"  BMR      : {result.raw.bmr_kcal or 'N/A (not on this model)'} kcal")
    print(f"  Right arm: {result.raw.right_arm.value} {result.raw.right_arm.unit} ({result.raw.right_arm.percent_of_ideal}%)")
    print(f"  Left arm : {result.raw.left_arm.value} {result.raw.left_arm.unit} ({result.raw.left_arm.percent_of_ideal}%)")
    print(f"  Trunk    : {result.raw.trunk.value} {result.raw.trunk.unit} ({result.raw.trunk.percent_of_ideal}%)")
    print(f"  Notes    : {result.raw.extraction_notes or 'none'}")
    print(f"  Phase ∠  : {result.raw.phase_angle or 'N/A'}")
    print(f"  WHR      : {result.raw.waist_hip_ratio or 'N/A'}")

    print("\n── Computed flags ──")
    print(f"  ARM_ASYMMETRY      : {result.flags.arm_asymmetry} (diff: {result.flags.arm_diff_grams}g, weaker: {result.flags.weaker_arm})")
    print(f"  LEG_ASYMMETRY      : {result.flags.leg_asymmetry} (diff: {result.flags.leg_diff_grams}g, weaker: {result.flags.weaker_leg})")
    print(f"  ELEVATED_BF        : {result.flags.elevated_bf}")
    print(f"  TRUNK_UNDERDEVELOPED: {result.flags.trunk_underdeveloped}")


Testing: ../samples/inbody2.jfif
FAILED at stage [quality_check]: Image is overexposed — no text is visible. Move away from direct light.

Testing: ../samples/inbody3.jfif

── Raw extraction ──
  Model    : 570
  Gender   : female
  SMM      : 24.6 kg
  BF%      : 35.0%
  BMR      : 1351 kcal
  Right arm: 2.08 kg (155.0%)
  Left arm : 2.08 kg (155.0%)
  Trunk    : 21.05 kg (182.1%)
  Notes    : none
  Phase ∠  : N/A
  WHR      : 0.89

── Computed flags ──
  ARM_ASYMMETRY      : False (diff: 0.0g, weaker: None)
  LEG_ASYMMETRY      : False (diff: 40.0g, weaker: left)
  ELEVATED_BF        : True
  TRUNK_UNDERDEVELOPED: False

Testing: ../samples/inbody4.jfif

── Raw extraction ──
  Model    : 270S
  Gender   : male
  SMM      : 105.8 lb
  BF%      : 6.1%
  BMR      : 2135 kcal
  Right arm: 19.0 lb (136.7%)
  Left arm : 19.8 lb (139.0%)
  Trunk    : 82.2 lb (120.5%)
  Notes    : none
  Phase ∠  : 7.5
  WHR      : N/A

── Computed flags ──
  ARM_ASYMMETRY      : True (diff: 362.9g, weaker